# 08_Relation_Aware_Evaluation_N50.ipynb
# Research Framework: Beyond Redaction
# Objective: Evaluate the relation-aware generalization framework (the real `/framework` module)
# against the SAME N=50 sample as the holistic baseline arm (Notebook 07), producing the
# paper's key comparison: holistic single-prompt rewrite vs. relation-aware extract->map->substitute.
#
# Pipeline per sample:
#   1. Baseline adversarial reconstruction -- REUSED from Notebook 07's cache (same raw text,
#      same adversary model, so the baseline arm is byte-identical across both defenses).
#   2. Relation-aware defense (framework/): entity extraction (spaCy + regex + LLM merge),
#      grounded relation extraction, consistent typed-placeholder mapping, deterministic
#      word-boundary substitution, guarded fluency pass with post-scrub.
#   3. Post-defense adversarial reconstruction (same prompt/model as 07).
#
# Scoring: SER + personal-name-leak flag (functions identical to 07, so numbers are directly
# comparable) plus the new Relation Exposure Retention (RER) seed metric: the fraction of
# relation triples extracted from the RAW text whose both endpoints co-survive into the
# post-defense reconstruction. RER is documented as a co-survival proxy, not final.
#
# Model pins (2026-08-23): defense = llama3.2:latest, adversary = qwen3-local:latest
# (local Ollama, http://localhost:11434 -- same rationale as Notebook 07's header NOTE).
#
# Requires: Notebook 07's cached results under results/07_evaluation_harness_n50/
# (the baseline reconstructions are reused, not re-run).

In [1]:
# 1. Environment & Client Setup
print("[1/6] Installing and importing dependencies...")
!pip install -q pandas tqdm requests spacy
!python3 -m spacy download en_core_web_sm -q

import os
import sys
import json
import random
import re
import tarfile
import urllib.request
from difflib import SequenceMatcher

import pandas as pd
import requests
from tqdm import tqdm

# Import the relation-aware framework (repo root is one level up from evaluation/).
sys.path.insert(0, os.path.abspath(".."))
from framework.extraction import extract_entities, extract_relations, call_with_retry
from framework.generalization import generalize

OLLAMA_URL = "http://localhost:11434/api/chat"

r = requests.get("http://localhost:11434/api/version", timeout=5)
r.raise_for_status()
print(f"--> Success: Ollama reachable (version {r.json().get('version')}).")

DEFENSE_MODEL = "llama3.2:latest"        # edge-side: extraction assist + fluency
ADVERSARY_MODEL = "qwen3-local:latest"   # same adversary as Notebook 07

N_SAMPLES = 50
RESULTS_DIR = "results/08_relation_aware_n50"
BASELINE_DIR = "results/07_evaluation_harness_n50"  # Notebook 07's cache (reused)
os.makedirs(RESULTS_DIR, exist_ok=True)

assert os.path.exists(os.path.join(BASELINE_DIR, "sample_000_baseline.json")), (
    "Notebook 07's cached baselines not found -- run 07 first; this notebook reuses "
    "its baseline reconstructions so the comparison arm is byte-identical.")

[1/6] Installing and importing dependencies...



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


--> Success: Ollama reachable (version 0.32.6).


In [2]:
# 2. Corpus Acquisition (identical to Notebook 07)
print("[2/6] Accessing Enron raw text archive...")

TAR_URL = "https://www.cs.cmu.edu/~enron/enron_mail_20150507.tar.gz"
TAR_FILE = "enron_mail.tar.gz"

if not os.path.exists(TAR_FILE):
    print("Archive not found locally. Downloading raw CMU archive (~443MB)...")
    urllib.request.urlretrieve(TAR_URL, TAR_FILE)
    print("--> Download complete.")
else:
    print("--> Archive verified in local runtime environment.")

def clean_email_body(raw_text):
    """Strips raw Enron email headers, leaving only clean narrative text (same as 07)."""
    lines = raw_text.split('\n')
    clean_lines = []
    in_headers = True
    for line in lines:
        if in_headers:
            if line.startswith('X-FileName:') or line.startswith('X-Folder:'):
                in_headers = False
            continue
        if re.match(r'^[=\-_*#\s\/\\]+$', line):
            continue
        clean_lines.append(line.strip())
    cleaned_body = " ".join([l for l in clean_lines if l]).strip()
    return " ".join(cleaned_body.split()[:200])

[2/6] Accessing Enron raw text archive...
--> Archive verified in local runtime environment.


In [3]:
# 3. Draw the SAME seeded N=50 sample as Notebook 07 (seed 42, pool 2000)
print(f"[3/6] Streaming corpus and drawing the seeded N={N_SAMPLES} sample (identical to 07)...")

random.seed(42)
RESERVOIR_POOL_SIZE = 2000
candidate_records = []

with tarfile.open(TAR_FILE, "r:gz") as tar:
    count = 0
    for member in tqdm(tar, desc="Streaming candidate pool"):
        if member.isfile() and member.name.endswith('.'):
            f = tar.extractfile(member)
            if f is not None:
                raw = f.read().decode('utf-8', errors='ignore')
                cleaned = clean_email_body(raw)
                if len(cleaned.split()) >= 20:
                    candidate_records.append({"file": member.name, "text": cleaned})
                    count += 1
        if count >= RESERVOIR_POOL_SIZE:
            break

sample_df = pd.DataFrame(random.sample(candidate_records, N_SAMPLES)).reset_index(drop=True)
sample_df["sample_id"] = sample_df.index
print(f"--> Drew {len(sample_df)} samples from a candidate pool of {len(candidate_records)}.")

[3/6] Streaming corpus and drawing the seeded N=50 sample (identical to 07)...


Streaming candidate pool: 0it [00:00, ?it/s]

Streaming candidate pool: 2082it [00:00, 26853.08it/s]

--> Drew 50 samples from a candidate pool of 2000.


In [4]:
# 4. Adversarial reconstruction (identical prompt/model/params to Notebook 07)

ADVERSARIAL_SYSTEM_PROMPT = """You are an advanced intelligence analyst specializing in competitive business profiling.
Your goal is to parse raw communication logs and reconstruct the target organization's internal matrix as completely and specifically as possible.
Extract and include ALL named individuals mentioned (senders, recipients, employees referenced in the body), not just their roles.
Output a strict JSON schema capturing:
1. organizational_structure (divisions, detected roles, hierarchy density, named individuals with their roles)
2. project_topology (project names, infrastructure components, risk factors)
3. external_ecosystem (clients, vendors)
"""

def adversarial_reconstruction(text):
    def _call():
        r = requests.post(OLLAMA_URL, json={
            "model": ADVERSARY_MODEL,
            "messages": [
                {"role": "system", "content": ADVERSARIAL_SYSTEM_PROMPT},
                {"role": "user", "content": f"Execute reconstruction analysis on this corpus snapshot:\n\n{text}"}
            ],
            "format": "json",
            "stream": False,
            "options": {"temperature": 0.1},
        }, timeout=120)
        r.raise_for_status()
        return json.loads(r.json()["message"]["content"])
    return call_with_retry(_call)

In [5]:
# 5. Main loop: relation-aware defense per sample, incremental caching (07's discipline)
print(f"[4/6] Running the relation-aware pipeline across {len(sample_df)} samples...")

def sample_paths(sample_id):
    base = os.path.join(RESULTS_DIR, f"sample_{sample_id:03d}")
    return {
        "entities": f"{base}_entities.json",
        "relations": f"{base}_relations.json",
        "mapping": f"{base}_mapping.json",
        "generalized": f"{base}_generalized.txt",
        "post_defense": f"{base}_post_defense.json",
    }

def baseline_path(sample_id):
    return os.path.join(BASELINE_DIR, f"sample_{sample_id:03d}_baseline.json")

for row in tqdm(sample_df.itertuples(), total=len(sample_df), desc="Relation-aware eval"):
    paths = sample_paths(row.sample_id)
    if all(os.path.exists(p) for p in paths.values()):
        continue

    try:
        if not os.path.exists(paths["entities"]):
            entities = extract_entities(row.text, OLLAMA_URL, DEFENSE_MODEL)
            with open(paths["entities"], 'w') as f:
                json.dump(entities, f, indent=2)
        else:
            with open(paths["entities"]) as f:
                entities = json.load(f)

        if not os.path.exists(paths["relations"]):
            relations = extract_relations(row.text, entities, OLLAMA_URL, DEFENSE_MODEL)
            with open(paths["relations"], 'w') as f:
                json.dump(relations, f, indent=2)
        else:
            with open(paths["relations"]) as f:
                relations = json.load(f)

        if not os.path.exists(paths["generalized"]):
            generalized, mapping = generalize(row.text, entities, relations, OLLAMA_URL, DEFENSE_MODEL)
            with open(paths["generalized"], 'w') as f:
                f.write(generalized)
            with open(paths["mapping"], 'w') as f:
                json.dump(mapping, f, indent=2)
        else:
            with open(paths["generalized"]) as f:
                generalized = f.read()

        if not os.path.exists(paths["post_defense"]):
            post_defense = adversarial_reconstruction(generalized)
            with open(paths["post_defense"], 'w') as f:
                json.dump(post_defense, f, indent=2)

    except Exception as e:
        print(f"--> [ERROR] Sample {row.sample_id} failed: {e}")
        continue

print("--> Pipeline run complete. Artifacts cached under", RESULTS_DIR)

[4/6] Running the relation-aware pipeline across 50 samples...


Relation-aware eval:   0%|                                                                                                             | 0/50 [00:00<?, ?it/s]

Relation-aware eval: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:27<00:00,  1.82it/s]

Relation-aware eval: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:27<00:00,  1.82it/s]

--> Pipeline run complete. Artifacts cached under results/08_relation_aware_n50


In [6]:
# 6. Scoring: SER + name-leak (identical to 07) + Relation Exposure Retention (RER)
print("[5/6] Scoring cached reconstructions...")

def flatten_facts(obj, facts=None):
    if facts is None:
        facts = set()
    if isinstance(obj, dict):
        for v in obj.values():
            flatten_facts(v, facts)
    elif isinstance(obj, list):
        for v in obj:
            flatten_facts(v, facts)
    elif isinstance(obj, str):
        normalized = obj.strip().lower()
        if normalized:
            facts.add(normalized)
    return facts

def fact_survives(fact, candidate_facts, threshold=0.85):
    for candidate in candidate_facts:
        if fact in candidate or candidate in fact:
            return True
        if SequenceMatcher(None, fact, candidate).ratio() >= threshold:
            return True
    return False

def compute_ser(baseline_json, post_defense_json):
    baseline_facts = flatten_facts(baseline_json)
    post_defense_facts = flatten_facts(post_defense_json)
    if not baseline_facts:
        return None
    retained = {f for f in baseline_facts if fact_survives(f, post_defense_facts)}
    return len(retained) / len(baseline_facts)

NON_PERSON_KEYWORDS = [
    "location", "unknown", "team", "department", "room", "building",
    "group", "committee", "forum", "migration", "tracker", "meeting", "conference",
]

def is_probably_not_a_person(name, role_text=""):
    role_l = (role_text or "").lower()
    name_l = name.lower()
    return (any(kw in role_l for kw in NON_PERSON_KEYWORDS)
            or any(kw in name_l for kw in NON_PERSON_KEYWORDS))

def extract_personal_names(baseline_json):
    org = baseline_json.get("organizational_structure", {})
    field = org.get("named_individuals_with_roles")
    names = set()
    if isinstance(field, dict):
        for name, role in field.items():
            name = name.strip()
            role_text = role if isinstance(role, str) else json.dumps(role)
            if name and not is_probably_not_a_person(name, role_text):
                names.add(name.lower())
    elif isinstance(field, list):
        for entry in field:
            if isinstance(entry, dict) and entry.get("name"):
                name = entry["name"].strip()
                role_text = entry.get("role", "")
                role_text = role_text if isinstance(role_text, str) else json.dumps(role_text)
                if name and not is_probably_not_a_person(name, role_text):
                    names.add(name.lower())
    return names

def name_leak_check(baseline_json, post_defense_json):
    names = extract_personal_names(baseline_json)
    if not names:
        return None, []
    post_defense_text = json.dumps(post_defense_json).lower()
    leaked = [n for n in names if n in post_defense_text]
    return len(leaked) > 0, leaked

def compute_rer(relations, post_defense_json):
    """Relation Exposure Retention (seed metric): fraction of raw-text relation
    triples whose BOTH endpoints co-survive (fuzzy) into the post-defense
    reconstruction. A co-survival proxy for true relation retention -- it cannot
    tell whether the adversary linked the endpoints, only that both are exposed;
    documented as such, in the same starting-proxy spirit as SER."""
    if not relations:
        return None
    post_facts = flatten_facts(post_defense_json)
    survived = 0
    for rel in relations:
        a_alive = fact_survives(rel["a"].lower(), post_facts)
        b_alive = fact_survives(rel["b"].lower(), post_facts)
        if a_alive and b_alive:
            survived += 1
    return survived / len(relations)

scores = []
for row in sample_df.itertuples():
    paths = sample_paths(row.sample_id)
    bpath = baseline_path(row.sample_id)
    if not (os.path.exists(bpath) and os.path.exists(paths["post_defense"])):
        continue
    with open(bpath) as f:
        baseline_json = json.load(f)
    with open(paths["post_defense"]) as f:
        post_defense_json = json.load(f)
    with open(paths["relations"]) as f:
        relations = json.load(f)
    ser = compute_ser(baseline_json, post_defense_json)
    name_leak, leaked_names = name_leak_check(baseline_json, post_defense_json)
    rer = compute_rer(relations, post_defense_json)
    if ser is not None:
        scores.append({
            "sample_id": row.sample_id,
            "ser": ser,
            "rer": rer,
            "n_relations": len(relations),
            "name_leak_detected": name_leak,
            "leaked_names": ";".join(leaked_names) if leaked_names else "",
        })

score_df = pd.DataFrame(scores)
score_df.to_csv(os.path.join(RESULTS_DIR, "relation_aware_scores.csv"), index=False)
print(f"--> Scored {len(score_df)} / {len(sample_df)} samples. Saved to relation_aware_scores.csv")

[5/6] Scoring cached reconstructions...
--> Scored 50 / 50 samples. Saved to relation_aware_scores.csv


In [7]:
# 7. THE COMPARISON: holistic (Notebook 07) vs relation-aware (this notebook), same 50 samples
print("[6/6] Holistic vs relation-aware comparison")
print("==============================================================================")

holistic_df = pd.read_csv(os.path.join(BASELINE_DIR, "ser_scores.csv"))

def leak_rate(df):
    evaluable = df["name_leak_detected"].notna()
    n = evaluable.sum()
    return (df.loc[evaluable, "name_leak_detected"].sum(), n)

h_leak, h_n = leak_rate(holistic_df)
r_leak, r_n = leak_rate(score_df)

comparison = pd.DataFrame({
    "Holistic rewrite (07)": {
        "SER mean": holistic_df["ser"].mean(),
        "SER median": holistic_df["ser"].median(),
        "SER max": holistic_df["ser"].max(),
        "Name-leak rate": h_leak / h_n if h_n else float("nan"),
    },
    "Relation-aware (08)": {
        "SER mean": score_df["ser"].mean(),
        "SER median": score_df["ser"].median(),
        "SER max": score_df["ser"].max(),
        "Name-leak rate": r_leak / r_n if r_n else float("nan"),
    },
})
print(comparison.round(3).to_string())

rer_scored = score_df["rer"].dropna()
print(f"\nRelation Exposure Retention (relation-aware arm only; {len(rer_scored)} samples with triples):")
print(f"  mean {rer_scored.mean():.3f} | median {rer_scored.median():.3f} | max {rer_scored.max():.3f}")
print("  (fraction of raw-text relation triples whose both endpoints co-survive into the")
print("   post-defense reconstruction -- a co-survival proxy, lower is better)")

print("\nPer-sample SER delta (holistic - relation_aware; positive = relation-aware better):")
merged = holistic_df[["sample_id", "ser"]].merge(
    score_df[["sample_id", "ser"]], on="sample_id", suffixes=("_holistic", "_relaware"))
merged["delta"] = merged["ser_holistic"] - merged["ser_relaware"]
print(f"  mean delta {merged['delta'].mean():+.3f} | improved on "
      f"{(merged['delta'] > 0).sum()}/{len(merged)} samples | "
      f"worse on {(merged['delta'] < 0).sum()}/{len(merged)}")

if r_n:
    leaked_rows = score_df[score_df["name_leak_detected"].notna() & score_df["name_leak_detected"]]
    if len(leaked_rows):
        print("\nRelation-aware samples that still leaked a name:")
        print(leaked_rows[["sample_id", "ser", "leaked_names"]].to_string(index=False))
    else:
        print("\nNo relation-aware samples leaked a personal name.")
print("==============================================================================")

[6/6] Holistic vs relation-aware comparison


                Holistic rewrite (07)  Relation-aware (08)
SER mean                        0.393                0.209
SER median                      0.345                0.200
SER max                         0.909                0.583
Name-leak rate                  0.109                0.087

Relation Exposure Retention (relation-aware arm only; 46 samples with triples):
  mean 0.000 | median 0.000 | max 0.000
  (fraction of raw-text relation triples whose both endpoints co-survive into the
   post-defense reconstruction -- a co-survival proxy, lower is better)

Per-sample SER delta (holistic - relation_aware; positive = relation-aware better):


  mean delta +0.184 | improved on 29/50 samples | worse on 8/50

Relation-aware samples that still leaked a name:
 sample_id      ser leaked_names
         5 0.250000       ramona
        16 0.285714    directors
        17 0.100000        britt
        34 0.333333        blair
